# Tracers carried into Seattle waterfront streets

A 12 m initial surge on a 6 m GeoClaw grid carries 3 m massless tracers from shallow offshore water into gaps between buildings


In [ ]:
%matplotlib inline
from pylab import *
from IPython.display import HTML
from matplotlib.animation import FuncAnimation
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
from clawpack.geoclaw import fgout_tools, topotools
from scipy.interpolate import RegularGridInterpolator
from shapely.geometry import Point, LineString
from shapely.ops import unary_union, transform, nearest_points
from shapely.strtree import STRtree
import numpy as np
import rasterio
import shapely
import sys
import builtins
sys.path.insert(0, '../../src')
from geoclaw_debris import debris_tracking
from geoclaw_debris.ship_tracking import LocalProjection


In [ ]:
from netCDF4 import Dataset
from prepare_topo import file_hash
FLOW_FILE = 'pike_place_surge12_fgout.nc'
with Dataset(FLOW_FILE) as ds:
    assert ds.terrain_sha256 == file_hash('seattle_buildings_mhw.tt3'), 'Flow terrain is stale: rerun make flow.'
    assert ds.surge_height_m == 12.0 and ds.grid_spacing_m == 6.0, 'Expected the 12 m surge on a 6 m grid.'
    assert ds.dimensions['time'].size == 181, 'Expected a 900 s run sampled every 5 s.'
    B_xy = array(ds['B'][0])
x, y, t, qoi_arrays = fgout_tools.read_netcdf_arrays(FLOW_FILE, qois=['h', 'u', 'v'], verbose=False)
print(f'flow grid: {len(x)} x {len(y)}; {len(t)} frames from {t[0]:.0f} to {t[-1]:.0f} s')
X, Y = meshgrid(x, y, indexing='ij')
bare_topo = topotools.Topography('seattle_19arcsec_mhw.tt3', topo_type=3)
bare_grid = RegularGridInterpolator((bare_topo.x, bare_topo.y), bare_topo.Z.T)(column_stack((X.ravel(), Y.ravel()))).reshape(X.shape)
building_grid = (B_xy - bare_grid) >= 10.0
depth = qoi_arrays['h'].transpose((1, 2, 0))
u_ms = qoi_arrays['u'].transpose((1, 2, 0))
v_ms = qoi_arrays['v'].transpose((1, 2, 0))


In [ ]:
lat_mid = 0.5 * (y[0] + y[-1])
m_per_deg_lat = 111000.
m_per_deg_lon = 111000. * cos(radians(lat_mid))
aspect = 1 / cos(radians(lat_mid))

u_degs = u_ms / m_per_deg_lon
v_degs = v_ms / m_per_deg_lat

def make_fcn(qoi):
    interp = RegularGridInterpolator((x, y, t), qoi, method='linear', fill_value=nan, bounds_error=False)
    return lambda xi, yi, ti: interp((xi, yi, ti))

depth_fcn = make_fcn(depth)
u_fcn = make_fcn(u_degs)
v_fcn = make_fcn(v_degs)

# use the same occupied grid cells for both the drawing and tracer collisions
dx, dy = x[1] - x[0], y[1] - y[0]
bx, by = X[building_grid], Y[building_grid]
building_union = shapely.union_all(shapely.box(bx-dx/2, by-dy/2, bx+dx/2, by+dy/2))
building_polys = list(building_union.geoms) if hasattr(building_union, 'geoms') else [building_union]
print(f'{building_grid.sum()} occupied flow cells in {len(building_polys)} connected building regions')


In [ ]:
# kep only wet points where the first wave drives water toward land.
t0 = 20.
dt = 1.0
frame0 = int(argmin(abs(t-t0)))
avg_deg_per_m = 0.5 * (1/m_per_deg_lon + 1/m_per_deg_lat)
L = 3.0 * avg_deg_per_m
projection = LocalProjection(float(x.mean()), float(y.mean()))
metric_buildings = [transform(projection.to_xy, poly) for poly in building_polys]
metric_building_union = unary_union(metric_buildings)

debris_list = []
z0_list = []
for i in range(0, len(x), 3):
    for j in range(0, len(y), 3):
        lon, lat = x[i], y[j]
        if not (-122.356 < lon < -122.340 and 47.606 < lat < 47.616):
            continue
        if not (-8.0 <= B_xy[i,j] <= 0.0):
            continue
        inland_speed = (u_ms[i,j,frame0] + v_ms[i,j,frame0]) / sqrt(2.)
        if depth[i,j,frame0] <= 0.5 or inland_speed <= 0.4:
            continue
        if Point(*projection.to_xy(lon, lat)).distance(metric_building_union) >= 80.:
            continue
        db = debris_tracking.DebrisObject()
        db.L = [L, L, L]
        db.phi = [pi/2, pi/2, pi/2]
        db.z = (lon-L/2, lat-L/2, 0.0)
        db.advect = True
        db.rho = 0.0
        if Point(lon, lat).distance(building_union) > db.radius:
            debris_list.append(db)
            z0_list.append(db.z)

print(f'released {len(debris_list)} tracers in shallow offshore water')
assert debris_list, 'No suitable releases found'


In [ ]:
import time

nsteps = int((t[-1] - t0) / dt)
domain = [x.min(), x.max(), y.min(), y.max()]
start_clock = time.time()
debris_path_list = []
for i, (db, z0) in enumerate(zip(debris_list, z0_list)):
    path = debris_tracking.make_massless_tracer_path(db, z0, building_union, domain, t0, dt, nsteps, depth_fcn, u_fcn, v_fcn)
    debris_path_list.append(path)
    if (i+1) % 25 == 0:
        print(f'{i+1}/{len(debris_list)} paths complete')

path_x = array([p.x_path.mean(axis=1) for p in debris_path_list])
path_y = array([p.y_path.mean(axis=1) for p in debris_path_list])
assert isfinite(path_x).all() and isfinite(path_y).all()
assert not shapely.contains_xy(building_union, path_x, path_y).any()
assert not builtins.any(LineString(column_stack((xi, yi))).crosses(building_union) for xi, yi in zip(path_x, path_y))
path_dist = hypot((path_x-path_x[:,[0]])*m_per_deg_lon, (path_y-path_y[:,[0]])*m_per_deg_lat)

# using the original bare-ground raster to distinguish flooded streets from piers
with rasterio.open('../../SeattleBuildings/seattle_19arcsec_navd88_wgs84.tif') as src:
    bare_end_m = array([value[0]-2.484 for value in src.sample(zip(path_x[:,-1], path_y[:,-1]))])

# a street endpoint has land above 2 m and building footprints on opposing sides
metric_tree = STRtree(metric_buildings)
between_buildings = []
for lon, lat, bare in zip(path_x[:,-1], path_y[:,-1], bare_end_m):
    point = Point(*projection.to_xy(lon, lat))
    directions = []
    for k in metric_tree.query(point.buffer(30)):
        building = metric_buildings[k]
        if building.distance(point) <= 30.:
            nearest = nearest_points(point, building)[1]
            vector = array(nearest.coords[0])-array(point.coords[0])
            directions.append(vector/(np.linalg.norm(vector)+1e-12))
    opposite_sides = builtins.any(dot(a,b) < -0.2 for k,a in enumerate(directions) for b in directions[k+1:])
    between_buildings.append(bare > 2. and opposite_sides)
between_buildings = array(between_buildings, dtype=bool)
assert between_buildings.sum() > 0, 'No tracer reached a street gap'

savez_compressed('pike_place_tracers_surge12.npz', x=path_x, y=path_y, times=debris_path_list[0].times, bare_end_m=bare_end_m, between_buildings=between_buildings)
print(f'done in {time.time()-start_clock:.1f} s: {sum(bare_end_m>2)}/{len(bare_end_m)} ended over land; {between_buildings.sum()} between buildings; median max travel {median(path_dist.max(axis=1)):.1f} m; no building crossings')


In [ ]:
from clawpack.visclaw import colormaps
cmap_water = colormaps.make_colormap({0:[0,1,1], 1:[0,0,1]})
cmap_buildings = ListedColormap(['#f4a340'])

# southern waterfront: several paths continue into gaps between buildings
x1, x2 = -122.348, -122.3385
y1, y2 = 47.607, 47.612
view_ix = where((x >= x1) & (x <= x2))[0]
view_iy = where((y >= y1) & (y <= y2))[0]
view_index = ix_(view_ix, view_iy)
X_view, Y_view = X[view_index], Y[view_index]
B_view = B_xy[view_index]
buildings_view = building_grid[view_index]

fig, ax = subplots(figsize=(12, 7))
ax.set_facecolor('#f2f1ed')  # dry land
initial_depth = depth[:, :, frame0][view_index]
water = ax.pcolormesh(X_view, Y_view, ma.masked_where(initial_depth < 0.001, initial_depth), cmap=cmap_water, vmin=0, vmax=12, shading='nearest')
ax.pcolormesh(X_view, Y_view, ma.masked_where(~buildings_view, buildings_view), cmap=cmap_buildings, vmin=0, vmax=1, shading='nearest', zorder=3)
ax.legend(handles=[Patch(facecolor='#f4a340', label='building cells (6 m grid)')], loc='upper right')
ax.contour(X_view, Y_view, B_view, levels=[0], colors='black', linewidths=0.8)

dist_color_max = max(25., 25.*ceil(percentile(path_dist, 99)/25.))
dots = ax.scatter(path_x[:,0], path_y[:,0], c=path_dist[:,0], cmap='plasma', s=24, linewidths=0.8, edgecolors='white', vmin=0, vmax=dist_color_max, zorder=5)
trail_lines = [ax.plot([], [], color='white', lw=0.7, alpha=0.85, zorder=4)[0] for _ in debris_list]
fig.colorbar(water, ax=ax, label='water depth (m)', shrink=0.75, pad=0.02)
fig.colorbar(dots, ax=ax, label='tracer distance (m)', shrink=0.75, pad=0.06)
ax.set_xlim(x1, x2)
ax.set_ylim(y1, y2)
ax.set_aspect(aspect)
ax.set_title(f'water and tracers at t = {t0:.0f} s')
ax.ticklabel_format(useOffset=False)
fig.tight_layout()

times = debris_path_list[0].times
def update(n):
    water_frame = int(argmin(abs(t - times[n])))
    assert abs(t[water_frame] - times[n]) < 1e-6
    current_depth = depth[:, :, water_frame][view_index]
    water.set_array(ma.masked_where(current_depth < 0.001, current_depth).ravel())
    dots.set_offsets(column_stack((path_x[:,n], path_y[:,n])))
    dots.set_array(path_dist[:,n])
    for i, line in enumerate(trail_lines):
        first = max(0, n-30)
        line.set_data(path_x[i,first:n+1], path_y[i,first:n+1])
    ax.set_title(f'water and tracers at t = {times[n]:.0f} s')

anim = FuncAnimation(fig, update, frames=range(0, len(times), 20), interval=170)
rcParams['animation.embed_limit'] = 50
animation_html = HTML(anim.to_jshtml())
close(fig)
animation_html


In [ ]:
# full trajectories and final street endpoints in two waterfront neighborhoods
views = [(-122.356, -122.350, 47.612, 47.616), (-122.349, -122.339, 47.607, 47.612)]
fig_streets, axes = subplots(1, 2, figsize=(14, 6))
for ax, (left, right, bottom, top) in zip(axes, views):
    ix = where((x >= left) & (x <= right))[0]
    iy = where((y >= bottom) & (y <= top))[0]
    ax.pcolormesh(x[ix], y[iy], B_xy[np.ix_(ix, iy)].T, cmap='terrain', vmin=-10, vmax=20, shading='auto')
    ax.pcolormesh(x[ix], y[iy], ma.masked_where(~building_grid[np.ix_(ix, iy)].T, building_grid[np.ix_(ix, iy)].T), cmap=cmap_buildings, vmin=0, vmax=1, shading='nearest', zorder=3)
    for k in range(len(path_x)):
        if ((path_x[k] > left) & (path_x[k] < right) & (path_y[k] > bottom) & (path_y[k] < top)).any():
            ax.plot(path_x[k], path_y[k], color='crimson' if between_buildings[k] else 'white', lw=1.2 if between_buildings[k] else 0.65, alpha=0.95, zorder=4)
            ax.scatter(path_x[k,0], path_y[k,0], s=16, c='cyan', edgecolors='black', linewidths=0.4, zorder=5)
            if between_buildings[k]:
                ax.scatter(path_x[k,-1], path_y[k,-1], s=18, facecolors='lime', edgecolors='black', linewidths=0.6, zorder=6)
    ax.set(xlim=(left,right), ylim=(bottom,top))
    ax.set_aspect(1/cos(radians((bottom+top)/2)))
    ax.ticklabel_format(useOffset=False)
fig_streets.suptitle(f'12 m surge on a 6 m grid: {between_buildings.sum()} tracers finish between buildings (green)')
fig_streets.legend(handles=[Patch(facecolor='#f4a340', label='building cells (6 m grid)')], loc='upper right')
fig_streets.tight_layout()
fig_streets.savefig('tracer_streets.png', dpi=180)
fig_streets
